# Phase 0 + Phase 1 on Colab — sequencer for `IMPLEMENTATION_PLAN.md`

**All logic lives in the repo** (`embeded/` on `main`); this notebook only sequences the
commands on a GPU and checkpoints generated artifacts to an optional **Hugging Face dataset repo**.
Gates are defined in `IMPLEMENTATION_PLAN.md` §4.

**Setup, every session**
1. `Runtime ▸ Change runtime type ▸ T4 GPU`.
2. (Recommended) create Colab secrets `EMBEDED_HF_REPO_ID` and `HF_TOKEN` so cell 1 can pull/push checkpoints.
3. Run cells top to bottom. Each numbered cell = one plan step; **STOP** if a gate cell prints FAIL.
4. Sharing / collaboration rules: see `COLAB.md` in the repo.

⚠ Colab free tier: expect a disconnect at ~90 min idle / ~12 h hard limit — the HF artifact
checkpoint (cell 1 + final cell) makes reruns cheap, and `run_all` steps are individually resumable.


In [ ]:
# 0. clone the repo (or `git pull` if you iterate here)
%cd -q /content
!rm -rf EmbedEd && git clone -q https://github.com/ksu-gitreaper807/EmbedEd.git   # main
%cd -q /content/EmbedEd


In [ ]:
# 1. local caches + optional Hugging Face artifact sync (do this BEFORE any
#    `embeded.` import so settings picks up the env overrides)
import os, subprocess

ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'             # model + dataset cache
os.environ['EMBEDED_ARTIFACTS'] = ART                  # generated pipeline artifacts
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'

try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try:
            val = userdata.get(key)
        except Exception:
            val = None
        if val:
            os.environ.setdefault(key, val)
except Exception:
    pass

print('Artifacts dir:', os.environ['EMBEDED_ARTIFACTS'])
print('HF cache    :', os.environ['HF_HOME'])
print('HF repo     :', os.environ.get('EMBEDED_HF_REPO_ID', '<not configured; session starts fresh>'))


In [ ]:
# 2. pinned environment (versions from embeded/settings.py::PINNED — change both together)
# torch is deliberately NOT installed here: Colab's preinstalled build is already
# GPU-matched (torchvision/torchaudio included) and pinning it means a ~3 GB
# re-download every session. Never add `torch==` to this cell.
%pip install -q -U transformers==4.46.3 datasets==2.20.0 \
    sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 \
    scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2
import sys; sys.path.insert(0, '/content/EmbedEd')
# pip will print ERROR lines about packages this notebook never uses
# (google-genai/google-adk vs pydantic, gcsfs vs fsspec, diffusers vs
# huggingface-hub, hf-gradio vs gradio-client). Expected noise — the line
# below is the pass/fail signal:
import torch, transformers, datasets, sentence_transformers, sklearn, gradio
print(f"env OK: torch {torch.__version__} (cuda={torch.cuda.is_available()}) | transformers {transformers.__version__} | "
      f"datasets {datasets.__version__} | sentence-transformers {sentence_transformers.__version__} | "
      f"scikit-learn {sklearn.__version__} | gradio {gradio.__version__}")
import subprocess
subprocess.run(['python', '-m', 'scripts.hf_artifacts', 'pull', '--if-configured'], check=True)


**Phase 0.2 — load + verify counts** (gate G0: `9,126 lines / 8,063 unique fragments / 901,028 / 415,416 / 415,416` — the CodeXGLUE paper's 9,134 is the paper's number, not the file's; correction 9), **0.3 overlap** (§4.2), **0.4 token lengths** (fixes `MAX_LEN`). Numbers are auto-written to
`report/measurements.md`.


In [ ]:
%%time
!python -m embeded.data.prepare_data --hf --verify-spec


**Phase 0.5 — throughput test**: replaces the [illustrative] subset size / batch /
epochs (SCOPE P2-13) at the `MAX_LEN` fixed by 0.4 (512: p50 = 474 tokens). Batch counts
*triples* — the encoder sees 3× that many sequences per step — with the fp16-AMP recipe from
`settings.py` (plain fp32 OOMs the T4 at 256×32 already). A candidate that OOMs is reported
and skipped, not fatal. Takes ~4 min; commit the printed `SUGGESTED` block to
`embeded/settings.py` (repo is the source of truth) before mining/training.


In [ ]:
%%time
!python -m scripts.phase0_throughput --candidates 512:4 512:8 512:16 512:32 --minutes 0.5


**Phase 0.7 — s′ acquisition** (RQ3 blocker; correction 2/7). Dry-run first, then
real. If the probe can't confirm functionality labels on ~23 functionalities, **stop and
report the blocker** — do not improvise Route B (hand-labelling) mid-project.


In [ ]:
!python -m scripts.fetch_sprime --dry-run
!python -m scripts.fetch_sprime


**Offline correctness first**: the §7.2 adversarial exclusion test + pipeline tests
run in seconds — green before we burn GPU time.


In [ ]:
!python -m pytest -q embeded/tests


**Phase 1 mining**: encode all fragments (one GPU pass), mine C1/C2/C3 (same k, same
exclusion), then **gate G1: hardness check**. `FAIL` ⇒ fix mining; do NOT proceed to
training (SCOPE P1-6 / P6-3).


In [ ]:
%%time
!python -m embeded.mining.semantic_index --batch 32


In [ ]:
!python -m embeded.negatives --strategies random,bm25,semantic --k 20


In [ ]:
!python -m embeded.hardcheck
# exit code 0 = PASS; 1 = FAIL (hardcheck prints the means and which gap failed)


**Phase 1.5 — smoke run** (15 steps, no results implied): proves model load →
triples batch → loss → grad path on the real stack before Phase 2 wires `train.py`.


In [ ]:
import json, random, torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from embeded import settings as S

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(S.MODEL_ID); model = AutoModel.from_pretrained(S.MODEL_ID).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=S.LR)
rows = [json.loads(l) for l in open(S.ARTIFACTS / 'triples_C1.jsonl')]
random.Random(S.SEED).shuffle(rows); rows = rows[:400]

def enc(t):
    e = tok(t, padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
    h = model(**e).last_hidden_state; m = e['attention_mask'].unsqueeze(-1).float()
    return F.normalize((h*m).sum(1)/m.sum(1).clamp(min=1e-6), dim=1)

frags = {i: f['text'] for i, f in enumerate(map(json.loads, open(S.ARTIFACTS / 'fragments.jsonl')))}
model.train(); losses = []
for s0 in range(0, len(rows), 16):
    batch = rows[s0:s0+16]
    a = enc([frags[r['anchor']] for r in batch]); p = enc([frags[r['positive']] for r in batch])
    n = enc([frags[r['negative']] for r in batch])
    loss = F.triplet_margin_loss(a, p, n, margin=0.2)
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    if len(losses) == 15: break
print(f"steps={len(losses)} first={losses[0]:.3f} last={losses[-1]:.3f}")
assert losses[-1] < losses[0] or losses[0] < 1e-6, "loss not moving: harness problem"
model.eval()
with torch.no_grad():
    r = random.Random(S.SEED).sample(rows, 30)
    gap_pos = (enc([frags[x['anchor']] for x in r]) * enc([frags[x['positive']] for x in r])).sum(1).mean()
    gap_neg = (enc([frags[x['anchor']] for x in r]) * enc([frags[x['negative']] for x in r])).sum(1).mean()
print(f"after smoke: mean cos(anchor,positive)={gap_pos:.3f} vs (anchor,negative)={gap_neg:.3f}")
print("SMOKE OK" if gap_pos > gap_neg else "SMOKE WEIRD — check before Phase 2")


**End of session — push `report/measurements.md` and artifacts to Hugging Face, then optionally download a zip.**

The final cell uploads `EMBEDED_ARTIFACTS/` plus `report/measurements.md` to the dataset repo
named by `EMBEDED_HF_REPO_ID` (if configured). Keep tokens in Colab secrets, not in notebook cells.


In [ ]:
import subprocess
subprocess.run(['python', '-m', 'scripts.hf_artifacts', 'push', '--if-configured', '--include-report'], check=True)
!cd /content/EmbedEd && zip -q -r /content/measurements.zip report
from google.colab import files
files.download('/content/measurements.zip')
